In [1]:
!pip install pillow paho-mqtt

In [2]:
"""
main_gui.py — Interfaz principal del sistema
Integra: detección IA + planificación de ruta A* + comunicación MQTT al carrito
"""

import tkinter as tk
from tkinter import ttk, messagebox, simpledialog
import threading
import time
import json
import heapq
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Sin ventana emergente de matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.patches as patches
from PIL import Image, ImageTk
import tensorflow as tf
import paho.mqtt.client as mqtt

In [3]:
# ─── CONFIGURACIÓN GLOBAL ────────────────────────────────────────────────────
IMG_SIZE    = 224
CLASS_NAMES = ['picaras', 'chokosoda', 'oreo']

# Mapeo clase IA → waypoint (A/B/C)
CLASE_A_WAYPOINT = {
    'picaras'  : 'A',
    'chokosoda': 'B',
    'oreo'     : 'C',
}

CELL_PX   = 400
PASO      = 200
TAM_ROBOT = (200, 200)

# MQTT
BROKER   = "broker.hivemq.com"
PORT     = 1883
ROBOT_ID = "R00"
TOPIC_RUTA   = f"{ROBOT_ID}/teleop/ruta"
TOPIC_STATUS = f"{ROBOT_ID}/sensors/status"

In [5]:
# ─── FUNCIONES DE PÉRDIDA MODELO 1 ───────────────────────────────────────────
def iou_metric(y_true, y_pred):
    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)
    x1p, y1p, x2p, y2p = tf.unstack(y_pred, axis=-1)
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)
    area_g = tf.maximum((x2g - x1g) * (y2g - y1g), 0.0)
    area_p = tf.maximum((x2p - x1p) * (y2p - y1p), 0.0)
    union  = area_g + area_p - inter + 1e-7
    return tf.reduce_mean(inter / union)

def giou_loss(y_true, y_pred):
    x1p_raw, y1p_raw, x2p_raw, y2p_raw = tf.unstack(y_pred, axis=-1)
    x1p = tf.minimum(x1p_raw, x2p_raw); y1p = tf.minimum(y1p_raw, y2p_raw)
    x2p = tf.maximum(x1p_raw, x2p_raw); y2p = tf.maximum(y1p_raw, y2p_raw)
    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)
    area_g = (x2g - x1g) * (y2g - y1g)
    area_p = (x2p - x1p) * (y2p - y1p)
    union  = area_g + area_p - inter + 1e-7
    iou    = inter / union
    xc1 = tf.minimum(x1g, x1p);  yc1 = tf.minimum(y1g, y1p)
    xc2 = tf.maximum(x2g, x2p);  yc2 = tf.maximum(y2g, y2p)
    area_c = (xc2 - xc1) * (yc2 - yc1) + 1e-7
    giou = iou - (area_c - union) / area_c
    return tf.reduce_mean(1.0 - giou)

def mse_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

def bbox_combined_loss(y_true, y_pred):
    return mse_loss(y_true, y_pred) + giou_loss(y_true, y_pred)


# ─── FUNCIONES DE PÉRDIDA MODELO 2 ───────────────────────────────────────────
def iou_metric_2(y_true, y_pred):
    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)
    x1p, y1p, x2p, y2p = tf.unstack(y_pred, axis=-1)
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)
    area_g = tf.maximum((x2g - x1g) * (y2g - y1g), 0.0)
    area_p = tf.maximum((x2p - x1p) * (y2p - y1p), 0.0)
    union  = area_g + area_p - inter + 1e-7
    return tf.reduce_mean(inter / union)

def giou_loss_2(y_true, y_pred):
    x1p_raw, y1p_raw, x2p_raw, y2p_raw = tf.unstack(y_pred, axis=-1)
    x1p = tf.minimum(x1p_raw, x2p_raw); y1p = tf.minimum(y1p_raw, y2p_raw)
    x2p = tf.maximum(x1p_raw, x2p_raw); y2p = tf.maximum(y1p_raw, y2p_raw)
    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)
    area_g = (x2g - x1g) * (y2g - y1g)
    area_p = (x2p - x1p) * (y2p - y1p)
    union  = area_g + area_p - inter + 1e-7
    iou    = inter / union
    xc1 = tf.minimum(x1g, x1p);  yc1 = tf.minimum(y1g, y1p)
    xc2 = tf.maximum(x2g, x2p);  yc2 = tf.maximum(y2g, y2p)
    area_c = (xc2 - xc1) * (yc2 - yc1) + 1e-7
    giou = iou - (area_c - union) / area_c
    return tf.reduce_mean(1.0 - giou)

def mse_loss_2(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

def bbox_combined_loss_2(y_true, y_pred):
    mse  = mse_loss_2(y_true, y_pred)
    giou = giou_loss_2(y_true, y_pred)
    w_true = y_true[:, 2] - y_true[:, 0]; h_true = y_true[:, 3] - y_true[:, 1]
    w_pred = y_pred[:, 2] - y_pred[:, 0]; h_pred = y_pred[:, 3] - y_pred[:, 1]
    size_loss = tf.reduce_mean(tf.square(w_true - w_pred) + tf.square(h_true - h_pred))
    return mse + giou + 0.5 * size_loss


# ─── CARGA DE MODELOS ────────────────────────────────────────────────────────
print("[INFO] Cargando modelos...")
modelo1 = tf.keras.models.load_model(
    'mejor_modelo_mobilenet.keras',
    custom_objects={'bbox_combined_loss': bbox_combined_loss,
                    'mse_loss': mse_loss, 'giou_loss': giou_loss, 'iou_metric': iou_metric}
)
modelo2 = tf.keras.models.load_model(
    'mejor_modelo_mobilenet_rotacion.keras',
    custom_objects={'bbox_combined_loss': bbox_combined_loss_2,
                    'mse_loss': mse_loss_2, 'giou_loss': giou_loss_2, 'iou_metric': iou_metric_2}
)
print("[INFO] Modelos cargados correctamente.")


[INFO] Cargando modelos...
[INFO] Modelos cargados correctamente.


In [6]:
# ─── FUNCIONES DE PREDICCIÓN ─────────────────────────────────────────────────
def predict_combinado(input_batch):
    _, class_pred = modelo1.predict(input_batch, verbose=0)
    bbox_pred, _  = modelo2.predict(input_batch, verbose=0)
    x1 = np.minimum(bbox_pred[:, 0], bbox_pred[:, 2])
    y1 = np.minimum(bbox_pred[:, 1], bbox_pred[:, 3])
    x2 = np.maximum(bbox_pred[:, 0], bbox_pred[:, 2])
    y2 = np.maximum(bbox_pred[:, 1], bbox_pred[:, 3])
    bboxes    = np.stack([x1, y1, x2, y2], axis=-1)
    clase     = CLASS_NAMES[np.argmax(class_pred[0])]
    confianza = float(np.max(class_pred[0]))
    return bboxes[0], clase, confianza


In [7]:
# ─── LÓGICA DE POSICIÓN ───────────────────────────────────────────────────────
def determinar_posicion(bounding_box, ancho_imagen):
    """
    Retorna 'derecha' o 'izquierda' según el centro X del bounding box.
    bounding_box: [x1_norm, y1_norm, x2_norm, y2_norm] normalizados 0-1
    """
    cx = (bounding_box[0] + bounding_box[2]) / 2.0
    return 'derecha' if cx >= 0.5 else 'izquierda'

In [8]:
# ─── MAPA Y PLANIFICACIÓN ────────────────────────────────────────────────────
def celda_a_pixel(x_celda, y_celda):
    fila = 200 + CELL_PX * y_celda
    col  = 200 + CELL_PX * x_celda
    return (fila, col)

def pixel_a_metros(fila, col):
    x_m = round((col - 200) / 1000, 4)
    y_m = round((fila - 200) / 1000, 4)
    return [x_m, y_m]

def crear_mapa():
    g = 3; c = 400
    mapa = np.zeros((2800, 2800), dtype=np.uint8)
    mapa[0:g, :] = 1; mapa[-g:, :] = 1
    mapa[:, 0:g] = 1; mapa[:, -g:] = 1
    mapa[c*1:c*1+g, c*0:c*1]=1; mapa[c*1:c*1+g, c*2:c*3]=1; mapa[c*1:c*1+g, c*4:c*5]=1
    mapa[c*2:c*2+g, c*1:c*2]=1; mapa[c*2:c*2+g, c*3:c*6]=1
    mapa[c*3:c*3+g, c*0:c*1]=1; mapa[c*3:c*3+g, c*2:c*3]=1; mapa[c*3:c*3+g, c*4:c*5]=1; mapa[c*3:c*3+g, c*6:c*7]=1
    mapa[c*4:c*4+g, c*1:c*4]=1; mapa[c*5:c*5+g, c*0:c*1]=1; mapa[c*5:c*5+g, c*2:c*3]=1
    mapa[c*5:c*5+g, c*5:c*6]=1; mapa[c*6:c*6+g, c*3:c*4]=1; mapa[c*6:c*6+g, c*5:c*6]=1
    mapa[c*3:c*4, c*1:c*1+g]=1; mapa[c*6:c*7, c*1:c*1+g]=1; mapa[c*2:c*3, c*2:c*2+g]=1
    mapa[c*4:c*5, c*2:c*2+g]=1; mapa[c*1:c*2, c*3:c*3+g]=1; mapa[c*6:c*7, c*3:c*3+g]=1
    mapa[c*0:c*1, c*4:c*4+g]=1; mapa[c*2:c*3, c*4:c*4+g]=1; mapa[c*4:c*5, c*4:c*4+g]=1
    mapa[c*4:c*5, c*5:c*5+g]=1; mapa[c*1:c*2, c*6:c*6+g]=1; mapa[c*3:c*4, c*6:c*6+g]=1
    mapa[c*5:c*6, c*6:c*6+g]=1
    return mapa

class Nodo:
    def __init__(self, posicion, padre=None):
        self.posicion = posicion; self.padre = padre
        self.g = self.h = self.f = float('inf')
    def __eq__(self, o): return self.posicion == o.posicion
    def __lt__(self, o): return self.f < o.f

def heuristica(pos, obj):
    return np.sqrt((pos[0]-obj[0])**2 + (pos[1]-obj[1])**2)

def es_colision(mapa, x, y, tam):
    ancho, alto = tam
    if x <= 0 or x + ancho > mapa.shape[0] or y <= 0 or y + alto > mapa.shape[1]:
        return True
    h = int(ancho / 2)
    for i in range(h):
        for j in range(h):
            if mapa[x+i, y+j] != 0 or mapa[x-i, y-j] != 0:
                return True
    return False

def obtener_camino(nodo):
    camino, actual = [], nodo
    while actual:
        camino.append(actual.posicion); actual = actual.padre
    return camino[::-1]

def A_estrella(mapa, inicio, objetivo):
    nodos_abiertos = []; nodos_cerrados = set(); costos = {inicio: 0}
    n0 = Nodo(inicio); n0.g = 0; n0.h = heuristica(inicio, objetivo); n0.f = n0.h
    heapq.heappush(nodos_abiertos, n0)
    n_obj = Nodo(objetivo)
    while nodos_abiertos:
        actual = heapq.heappop(nodos_abiertos)
        nodos_cerrados.add(actual.posicion)
        if actual == n_obj:
            return obtener_camino(actual)
        x, y = actual.posicion
        for sx, sy in [(x-PASO,y),(x+PASO,y),(x,y-PASO),(x,y+PASO)]:
            if not es_colision(mapa, sx, sy, TAM_ROBOT):
                nuevo_costo = actual.g + 1
                if (sx,sy) not in nodos_cerrados:
                    if (sx,sy) not in costos or nuevo_costo < costos[(sx,sy)]:
                        costos[(sx,sy)] = nuevo_costo
                        nn = Nodo((sx,sy), actual); nn.g = nuevo_costo
                        nn.h = heuristica((sx,sy), objetivo); nn.f = nn.g + nn.h
                        heapq.heappush(nodos_abiertos, nn)
    return None

def planificar_ruta(mapa, inicio_px, waypoint_px, fin_px):
    tramo1 = A_estrella(mapa, inicio_px, waypoint_px)
    if tramo1 is None: return None
    tramo2 = A_estrella(mapa, waypoint_px, fin_px)
    if tramo2 is None: return None
    ruta_px     = tramo1 + tramo2[1:]
    ruta_metros = [pixel_a_metros(f, c) for f, c in ruta_px]
    return ruta_px, ruta_metros


In [9]:
# ─── COMUNICACIÓN MQTT ───────────────────────────────────────────────────────
def enviarDatosACarrito(ruta, on_done_callback):
    """Envía la ruta por MQTT y llama on_done_callback() cuando el carrito termina."""
    carrito_listo = threading.Event()

    def on_connect(client, userdata, flags, rc):
        if rc == 0:
            client.subscribe(TOPIC_STATUS)

    def on_message(client, userdata, msg):
        payload = msg.payload.decode().strip()
        if payload == "done":
            carrito_listo.set()

    client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER, PORT, keepalive=60)
    client.loop_start()
    time.sleep(1.5)

    payload = json.dumps({"ruta": ruta})
    client.publish(TOPIC_RUTA, payload)

    # Esperar hasta 5 minutos
    carrito_listo.wait(timeout=300)
    client.loop_stop()
    client.disconnect()
    on_done_callback()

def recibirDatosDeCarrito():
    """Alias — la recepción ocurre dentro de enviarDatosACarrito vía callback."""
    pass

In [10]:



# ════════════════════════════════════════════════════════════════════════════
# INTERFAZ PRINCIPAL
# ════════════════════════════════════════════════════════════════════════════
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Sistema de Detección y Navegación")
        self.configure(bg="#1a1a2e")
        self.resizable(True, True)

        # Estado interno
        self.puntos      = {}          # {'inicio': (xi,yi), 'fin': (xf,yf), 'A': ..., 'B': ..., 'C': ...}
        self.ultimo_frame     = None   # frame capturado
        self.ultimo_bbox      = None   # bbox normalizado
        self.ultima_clase     = None
        self.ultima_confianza = None
        self.mapa        = None
        self.ruta_metros = None
        self._cap        = None        # VideoCapture activo
        self._deteccion_activa = False

        self._build_ui()
        self.protocol("WM_DELETE_WINDOW", self._on_close)

    # ── Construcción de la UI ─────────────────────────────────────────────

    def _build_ui(self):
        AZUL    = "#16213e"
        ACENTO  = "#0f3460"
        VERDE   = "#00d4aa"
        TEXTO   = "#e0e0e0"
        ROJO    = "#e94560"
        BTN_FG  = "#ffffff"

        # ── Panel izquierdo (controles) ──
        left = tk.Frame(self, bg=AZUL, width=280)
        left.pack(side=tk.LEFT, fill=tk.Y, padx=10, pady=10)
        left.pack_propagate(False)

        tk.Label(left, text="🤖 Control", font=("Helvetica", 16, "bold"),
                 bg=AZUL, fg=VERDE).pack(pady=(20, 10))

        # Botón definir puntos
        self._btn_puntos = tk.Button(
            left, text="📍 Definir puntos",
            command=self._definir_puntos,
            bg=ACENTO, fg=BTN_FG, font=("Helvetica", 11, "bold"),
            relief=tk.FLAT, cursor="hand2", pady=8
        )
        self._btn_puntos.pack(fill=tk.X, padx=15, pady=6)

        # Botón detectar imagen
        self._btn_detectar = tk.Button(
            left, text="📷 Detectar imagen",
            command=self._iniciar_deteccion,
            bg=ACENTO, fg=BTN_FG, font=("Helvetica", 11, "bold"),
            relief=tk.FLAT, cursor="hand2", pady=8, state=tk.DISABLED
        )
        self._btn_detectar.pack(fill=tk.X, padx=15, pady=6)

        # Info detección
        self._lbl_clase = tk.Label(left, text="Clase: —",
                                   bg=AZUL, fg=TEXTO, font=("Helvetica", 10))
        self._lbl_clase.pack(pady=2)
        self._lbl_conf = tk.Label(left, text="Confianza: —",
                                  bg=AZUL, fg=TEXTO, font=("Helvetica", 10))
        self._lbl_conf.pack(pady=2)
        self._lbl_pos = tk.Label(left, text="Posición: —",
                                 bg=AZUL, fg=TEXTO, font=("Helvetica", 10))
        self._lbl_pos.pack(pady=2)

        ttk.Separator(left, orient='horizontal').pack(fill=tk.X, padx=15, pady=12)

        # Botón confirmar detección
        self._btn_confirmar = tk.Button(
            left, text="✅ Confirmar detección",
            command=self._confirmar_deteccion,
            bg="#1a5c3a", fg=BTN_FG, font=("Helvetica", 11, "bold"),
            relief=tk.FLAT, cursor="hand2", pady=8, state=tk.DISABLED
        )
        self._btn_confirmar.pack(fill=tk.X, padx=15, pady=6)

        self._btn_rechazar = tk.Button(
            left, text="🔄 Repetir detección",
            command=self._rechazar_deteccion,
            bg=ROJO, fg=BTN_FG, font=("Helvetica", 11, "bold"),
            relief=tk.FLAT, cursor="hand2", pady=8, state=tk.DISABLED
        )
        self._btn_rechazar.pack(fill=tk.X, padx=15, pady=6)

        ttk.Separator(left, orient='horizontal').pack(fill=tk.X, padx=15, pady=12)

        # Log de estado
        tk.Label(left, text="Estado del sistema:", bg=AZUL, fg=VERDE,
                 font=("Helvetica", 10, "bold")).pack(anchor=tk.W, padx=15)
        self._log = tk.Text(left, height=12, bg="#0d0d1a", fg=TEXTO,
                            font=("Courier", 9), relief=tk.FLAT, wrap=tk.WORD)
        self._log.pack(fill=tk.BOTH, padx=15, pady=6, expand=True)

        # ── Panel derecho (visualización) ──
        right = tk.Frame(self, bg="#1a1a2e")
        right.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=10, pady=10)

        # Canvas cámara / imagen capturada
        tk.Label(right, text="Vista de cámara / Captura",
                 bg="#1a1a2e", fg=VERDE, font=("Helvetica", 11, "bold")).pack()
        self._canvas_cam = tk.Canvas(right, bg="#0d0d1a", width=480, height=360,
                                     highlightthickness=1, highlightbackground=ACENTO)
        self._canvas_cam.pack(pady=(0, 10))

        # Canvas mapa
        tk.Label(right, text="Mapa y ruta generada",
                 bg="#1a1a2e", fg=VERDE, font=("Helvetica", 11, "bold")).pack()
        self._frame_mapa = tk.Frame(right, bg="#1a1a2e")
        self._frame_mapa.pack(fill=tk.BOTH, expand=True)

    # ── Log helper ────────────────────────────────────────────────────────

    def _log_msg(self, msg):
        self._log.insert(tk.END, f"▶ {msg}\n")
        self._log.see(tk.END)

    # ── PASO 1: Definir puntos ────────────────────────────────────────────

    def _definir_puntos(self):
        dlg = PuntosDialog(self)
        self.wait_window(dlg)
        if dlg.resultado:
            self.puntos = dlg.resultado
            self._log_msg(f"Inicio: {self.puntos['inicio']}  Fin: {self.puntos['fin']}")
            self._log_msg(f"A={self.puntos['A']}  B={self.puntos['B']}  C={self.puntos['C']}")
            self._btn_detectar.config(state=tk.NORMAL)
            self._log_msg("Puntos definidos. Listo para detectar.")
        else:
            self._log_msg("Definición de puntos cancelada.")

    # ── PASO 2: Detección ─────────────────────────────────────────────────

    def _iniciar_deteccion(self):
        conf = simpledialog.askfloat(
            "Umbral de confianza",
            "Ingresa el confidence_threshold (0.0 – 1.0):",
            minvalue=0.0, maxvalue=1.0, initialvalue=0.8, parent=self
        )
        if conf is None:
            return
        self._confidence_threshold = conf
        self._deteccion_activa = True
        self._btn_detectar.config(state=tk.DISABLED)
        self._btn_confirmar.config(state=tk.DISABLED)
        self._btn_rechazar.config(state=tk.DISABLED)
        self._log_msg(f"Iniciando cámara (threshold={conf:.2f})...")
        threading.Thread(target=self._loop_camara, daemon=True).start()

    def _loop_camara(self):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            self._log_msg("ERROR: no se pudo abrir la cámara.")
            self._btn_detectar.config(state=tk.NORMAL)
            return
        self._cap = cap

        while self._deteccion_activa:
            ret, frame = cap.read()
            if not ret:
                break

            orig_h, orig_w = frame.shape[:2]
            img_r = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            batch = np.expand_dims(img_r / 255.0, axis=0)

            bbox, clase, confianza = predict_combinado(batch)

            display = frame.copy()
            if confianza >= self._confidence_threshold:
                x1 = int(bbox[0] * orig_w); y1 = int(bbox[1] * orig_h)
                x2 = int(bbox[2] * orig_w); y2 = int(bbox[3] * orig_h)
                cv2.rectangle(display, (x1, y1), (x2, y2), (0, 212, 170), 2)
                cv2.putText(display, f"{clase}: {confianza:.2f}",
                            (x1, max(y1-10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 212, 170), 2)

                # Objeto detectado con suficiente confianza → capturar y parar
                self._deteccion_activa = False
                self.ultimo_frame     = display.copy()
                self.ultimo_bbox      = bbox
                self.ultima_clase     = clase
                self.ultima_confianza = confianza

                # Actualizar UI desde hilo principal
                self.after(0, self._mostrar_captura)
            else:
                self.after(0, lambda f=display: self._mostrar_frame(f))

        cap.release()
        self._cap = None

    def _mostrar_frame(self, frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(rgb).resize((480, 360))
        photo = ImageTk.PhotoImage(img)
        self._canvas_cam.photo = photo
        self._canvas_cam.create_image(0, 0, anchor=tk.NW, image=photo)

    def _mostrar_captura(self):
        self._mostrar_frame(self.ultimo_frame)

        posicion = determinar_posicion(self.ultimo_bbox, 1.0)

        self._lbl_clase.config(text=f"Clase: {self.ultima_clase}")
        self._lbl_conf.config(text=f"Confianza: {self.ultima_confianza:.2f}")
        self._lbl_pos.config(text=f"Posición: {posicion}")
        self._log_msg(f"Detectado: {self.ultima_clase} ({self.ultima_confianza:.2f}) — {posicion}")

        self._btn_confirmar.config(state=tk.NORMAL)
        self._btn_rechazar.config(state=tk.NORMAL)

    # ── PASO 3: Confirmación ──────────────────────────────────────────────

    def _confirmar_deteccion(self):
        posicion = determinar_posicion(self.ultimo_bbox, 1.0)

        if posicion == 'izquierda':
            self._log_msg("Objeto en izquierda — movimiento no se realizará.")
            messagebox.showwarning(
                "Posición inválida",
                "Clase detectada, pero el movimiento no se realizará porque el objeto está en la izquierda."
            )
            self._rechazar_deteccion()
            return

        # Posición derecha → continuar
        self._btn_confirmar.config(state=tk.DISABLED)
        self._btn_rechazar.config(state=tk.DISABLED)
        self._log_msg("Confirmado. Generando ruta...")
        self._generar_y_enviar_ruta()

    def _rechazar_deteccion(self):
        self._btn_confirmar.config(state=tk.DISABLED)
        self._btn_rechazar.config(state=tk.DISABLED)
        self._log_msg("Repetiendo detección...")
        self._iniciar_deteccion()

    # ── PASO 4-7: Ruta + MQTT ─────────────────────────────────────────────

    def _generar_y_enviar_ruta(self):
        # Seleccionar waypoint según clase
        waypoint_key = CLASE_A_WAYPOINT.get(self.ultima_clase)
        if waypoint_key is None or waypoint_key not in self.puntos:
            self._log_msg(f"ERROR: clase '{self.ultima_clase}' sin waypoint asignado.")
            return

        xi, yi = self.puntos['inicio']
        xf, yf = self.puntos['fin']
        wp     = self.puntos[waypoint_key]

        inicio_px   = celda_a_pixel(*( xi, yi ))
        fin_px      = celda_a_pixel(*( xf, yf ))
        waypoint_px = celda_a_pixel(*wp)

        self.mapa = crear_mapa().astype(np.float64)
        resultado = planificar_ruta(self.mapa, inicio_px, waypoint_px, fin_px)

        if resultado is None:
            self._log_msg("ERROR: no se encontró ruta válida.")
            messagebox.showerror("Error", "No se pudo generar la ruta completa.")
            return

        ruta_px, self.ruta_metros = resultado
        self._log_msg(f"Ruta generada: {len(self.ruta_metros)} puntos.")

        # Guardar JSON
        with open('ruta_generada.json', 'w') as f:
            json.dump({'ruta': self.ruta_metros}, f)

        # Mostrar mapa
        self._mostrar_mapa(ruta_px, inicio_px, waypoint_px, fin_px, waypoint_key)
        self._log_msg("Ruta generada, enviando datos al carrito...")

        # Enviar en hilo separado para no bloquear la UI
        threading.Thread(
            target=enviarDatosACarrito,
            args=(self.ruta_metros, self._ruta_terminada),
            daemon=True
        ).start()

    def _mostrar_mapa(self, ruta_px, inicio_px, waypoint_px, fin_px, waypoint_nombre):
        for w in self._frame_mapa.winfo_children():
            w.destroy()

        fig, ax = plt.subplots(figsize=(5, 5), facecolor='#0d0d1a')
        ax.set_facecolor('#0d0d1a')
        ax.imshow(self.mapa, cmap='Blues', origin='upper', alpha=0.6)
        ax.set_title(f"Ruta — waypoint {waypoint_nombre}", color='#00d4aa', fontsize=10)

        H, W = self.mapa.shape
        ax.set_xlim(0, W); ax.set_ylim(H, 0)
        ax.xaxis.set_major_locator(MultipleLocator(CELL_PX))
        ax.yaxis.set_major_locator(MultipleLocator(CELL_PX))
        ax.tick_params(colors='#888888', labelsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')
        ax.grid(True, which='major', linewidth=0.4, alpha=0.3, color='#555577')
        ax.set_aspect('equal')

        filas, cols = zip(*ruta_px)
        ax.plot(cols, filas, color='#00d4aa', linewidth=1.5, zorder=3)
        ax.plot(inicio_px[1],   inicio_px[0],   'gs', markersize=10, label='Inicio',  zorder=5)
        ax.plot(waypoint_px[1], waypoint_px[0], 'r^', markersize=10, label=waypoint_nombre, zorder=5)
        ax.plot(fin_px[1],      fin_px[0],      'm*', markersize=12, label='Fin',     zorder=5)
        ax.legend(fontsize=7, facecolor='#1a1a2e', labelcolor='#e0e0e0')

        plt.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=self._frame_mapa)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        plt.close(fig)

    def _ruta_terminada(self):
        self.after(0, self._on_ruta_terminada)

    def _on_ruta_terminada(self):
        self._log_msg("✅ Ruta terminada. Reiniciando detección...")
        messagebox.showinfo("Completado", "Ruta terminada")
        # Volver al paso de detección
        self._btn_detectar.config(state=tk.NORMAL)

    def _on_close(self):
        self._deteccion_activa = False
        if self._cap:
            self._cap.release()
        self.destroy()


# ════════════════════════════════════════════════════════════════════════════
# DIÁLOGO: Definir puntos
# ════════════════════════════════════════════════════════════════════════════
class PuntosDialog(tk.Toplevel):
    def __init__(self, parent):
        super().__init__(parent)
        self.title("Definir puntos")
        self.configure(bg="#16213e")
        self.resizable(False, False)
        self.resultado = None
        self._build()
        self.grab_set()

    def _build(self):
        AZUL  = "#16213e"; VERDE = "#00d4aa"; TEXTO = "#e0e0e0"; ACENTO = "#0f3460"

        tk.Label(self, text="Definir puntos del mapa",
                 bg=AZUL, fg=VERDE, font=("Helvetica", 13, "bold")).pack(pady=12)

        frame = tk.Frame(self, bg=AZUL)
        frame.pack(padx=20, pady=5)

        campos = [
            ("Punto INICIO (X, Y celda):", "inicio"),
            ("Punto FIN    (X, Y celda):", "fin"),
            ("Waypoint A   (X, Y celda):", "A"),
            ("Waypoint B   (X, Y celda):", "B"),
            ("Waypoint C   (X, Y celda):", "C"),
        ]
        self._entries = {}
        defaults = {"inicio": "0,0", "fin": "6,6", "A": "0,2", "B": "3,1", "C": "0,3"}

        for label, key in campos:
            row = tk.Frame(frame, bg=AZUL)
            row.pack(fill=tk.X, pady=3)
            tk.Label(row, text=label, bg=AZUL, fg=TEXTO,
                     font=("Helvetica", 10), width=30, anchor=tk.W).pack(side=tk.LEFT)
            e = tk.Entry(row, bg="#0d0d1a", fg=TEXTO, insertbackground=VERDE,
                         font=("Courier", 10), width=10, relief=tk.FLAT)
            e.insert(0, defaults[key])
            e.pack(side=tk.LEFT, padx=5)
            self._entries[key] = e

        tk.Label(self, text="Formato: X,Y  (número de celda, sin paréntesis)",
                 bg=AZUL, fg="#888888", font=("Helvetica", 8)).pack(pady=(0, 8))

        btn_frame = tk.Frame(self, bg=AZUL)
        btn_frame.pack(pady=10)
        tk.Button(btn_frame, text="Confirmar", command=self._confirmar,
                  bg="#1a5c3a", fg="white", font=("Helvetica", 10, "bold"),
                  relief=tk.FLAT, padx=16, pady=6, cursor="hand2").pack(side=tk.LEFT, padx=8)
        tk.Button(btn_frame, text="Cancelar", command=self.destroy,
                  bg="#5c1a1a", fg="white", font=("Helvetica", 10, "bold"),
                  relief=tk.FLAT, padx=16, pady=6, cursor="hand2").pack(side=tk.LEFT, padx=8)

    def _limpiar(self, texto):
        texto = texto.strip().replace('(','').replace(')','').replace(' ','')
        partes = [int(v) for v in texto.split(',')]
        return tuple(partes)

    def _confirmar(self):
        try:
            resultado = {}
            for key, entry in self._entries.items():
                resultado[key] = self._limpiar(entry.get())
            self.resultado = resultado
            self.destroy()
        except Exception:
            messagebox.showerror("Error", "Formato inválido. Usa X,Y (ej: 0,0)", parent=self)


In [11]:

# ─── ENTRADA ─────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    app = App()
    app.mainloop()

C:\Users\Apleppli\AppData\Local\Temp\ipykernel_39668\2732557473.py:15: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()
